# MÓDULO 3
## Tema 2. Principios de POO: encapsulamiento, herencia, polimorfismo y abstracción

**Objetivos:**
- Comprender y aplicar **encapsulamiento, herencia, polimorfismo y abstracción**.
- Usar propiedades para proteger invariantes sin perder ergonomía.
- Identificar cuándo conviene heredar y cuándo conviene componer.

## Índice
- Encapsulamiento (convenciones, invariantes y `@property`)
- Herencia (`is-a`), sobreescritura y `super()`
- Polimorfismo (duck typing y contratos)
- Abstracción y clases abstractas (`abc`)
- Trampas comunes y buenas prácticas

## Encapsulamiento

Encapsular es **controlar el acceso** al estado interno y ofrecer una interfaz estable.

En Python no hay “private” estricto, pero hay convenciones:
- `_atributo`: “uso interno”
- `__atributo`: *name mangling* (útil para evitar colisiones en herencia)

In [ ]:
class SensorTemperatura:
    def __init__(self):
        self._celsius = 20.0

    def leer(self):
        return self._celsius

    def calibrar(self, delta):
        self._celsius += float(delta)

s = SensorTemperatura()

# Acceso "correcto" mediante métodos
print("Lectura:", s.leer())
s.calibrar(1.5)
print("Lectura:", s.leer())

# Acceso directo al atributo (NO recomendado, pero posible)
s._celsius = 100
print(s._celsius)

# Acceso directo al atributo sin _ (NO recomendado, no posible)
# s.celsius = 100  # Esto generaría un error, ya que el atributo no existe sin el guion bajo

Lectura: 20.0
Lectura: 21.5
100


### `@property` para validar y mantener invariantes

Permite que el usuario escriba `obj.edad = 31` pero tú controlas qué ocurre.

In [5]:
class Persona:
    def __init__(self, nombre, edad):
        self._nombre = nombre
        self._edad = edad

    @property
    def nombre(self):
        print("Estoy en el getter de nombre")
        return self._nombre
    
    @nombre.setter
    def nombre(self, value):
        print("Estoy en el setter de nombre")
        self._nombre = value

    @property
    def edad(self):
        print("Estoy en el getter de edad")
        return self._edad

    @edad.setter
    def edad(self, value):
        print("Estoy en el setter de edad")
        if value < 0:
            raise ValueError("La edad no puede ser negativa")
        self._edad = value

p = Persona(nombre="Sara", edad=30)

print(f"{p.nombre} tiene {p.edad} años")   # getter
p.edad = 31     # setter
print(f"{p.nombre} tiene {p.edad} años")   # getter

# Chequeando el fallo
try:
    p.edad = -5    # ValueError
except ValueError as e:
    print("Error:", e)

Estoy en el getter de nombre
Estoy en el getter de edad
Sara tiene 30 años
Estoy en el setter de edad
Estoy en el getter de nombre
Estoy en el getter de edad
Sara tiene 31 años
Estoy en el setter de edad
Error: La edad no puede ser negativa


## Herencia

Herencia describe una relación “**es un**”:

- `Vehiculo` → `Moto`, `Coche`
- `Animal` → `Perro`, `Gato`

Cuando solo quieres reutilizar funcionalidad sin “es un”, suele ser mejor **composición** (Tema 4).

In [6]:
class Vehiculo:
    def __init__(self, marca):
        self.marca = marca

    def arrancar(self):
        return f"{self.marca}: arrancando"

class Moto(Vehiculo):
    def caballito(self):
        return f"{self.marca}: haciendo un caballito"

m = Moto(marca="Yamaha")
print(m.arrancar())
print(m.caballito())

Yamaha: arrancando
Yamaha: haciendo un caballito


### `super()`

`super()` permite reutilizar lógica de la clase base y reducir duplicación.

In [ ]:
class Empleado:
    def __init__(self, nombreEmpleado, salarioEmpleado):
        self.nombre = nombreEmpleado
        self.salario = float(salarioEmpleado)

    def info(self):
        return f"{self.nombre} cobra {self.salario:.2f}€"

class Manager(Empleado):
    def __init__(self, nombreManager, salarioManager, equipoManger):
        super().__init__(nombreEmpleado=nombreManager, salarioEmpleado=salarioManager)
        self.equipo = list(equipoManger)

    def info(self):
        base = super().info()
        return f"{base} y gestiona {len(self.equipo)} personas"

mgr = Manager(nombreManager="Ana", salarioManager=42000, equipoManger=["Luis", "Marta"])
print(mgr.info())

Pero tambien podemos simplificar los nombres

In [ ]:
class Empleado:
    def __init__(self, nombre, salario):
        self.nombre = nombre
        self.salario = float(salario)

    def info(self):
        return f"{self.nombre} cobra {self.salario:.2f}€"

class Manager(Empleado):
    def __init__(self, nombre, salario, equipo):
        super().__init__(nombre=nombre, salario=salario)
        self.equipo = list(equipo)

    def info(self):
        base = super().info()
        return f"{base} y gestiona {len(self.equipo)} personas"

mgr = Manager(nombre="Ana", salario=42000, equipo=["Luis", "Marta"])
print(mgr.info())

## Polimorfismo

Mismo “mensaje” (método) con distintas implementaciones.

En Python se apoya mucho en *duck typing*:
> “Si se comporta como un pato, es un pato.”

In [ ]:
class Email:
    def enviar(self, mensaje):
        print(f"Enviando email: {mensaje}")

class SMS:
    def enviar(self, mensaje):
        print(f"Enviando SMS: {mensaje}")

class WhatsApp:
    def enviar(self, mensaje):
        print(f"Enviando WhatsApp: {mensaje}")

def notificar(canales, mensaje):
    for canal in canales:
        canal.enviar(mensaje)

notificar(
    [Email(), SMS(), WhatsApp()],
    "Tu pedido ha sido enviado"
)

Enviando email: Tu pedido ha sido enviado
Enviando SMS: Tu pedido ha sido enviado
Enviando WhatsApp: Tu pedido ha sido enviado


## Abstracción

Abstraer es **quedarse con lo esencial** y ocultar detalles.

Cuando quieres imponer un contrato (“tienes que implementar `cobrar`”), usa `abc`.

In [1]:
# Ejemplo de NO usar abstracción

class PagoTarjeta:
    def cobrar(self, cantidad):
        return f"Cobrado {float(cantidad):.2f}€ con tarjeta"

class PagoBizum:
    def cobrar(self, cantidad):
        return f"Cobrado {float(cantidad):.2f}€ con Bizum"

class PagoEfectivo:
    pass  # alguien se olvida de implementar cobrar

def procesar(pago, cantidad):
    return pago.cobrar(cantidad)

print(procesar(PagoTarjeta(), 10))
print(procesar(PagoBizum(), 10))
print(procesar(PagoEfectivo(), 10)) # Error en tiempo de ejecución

Cobrado 10.00€ con tarjeta
Cobrado 10.00€ con Bizum


AttributeError: 'PagoEfectivo' object has no attribute 'cobrar'

In [ ]:
# Ejemplo de SI usar abstracción

from abc import ABC, abstractmethod

class Pago(ABC):
    @abstractmethod
    def cobrar(self, cantidad):
        pass

# ---------- IMPLEMENTACIONES CORRECTAS ----------

class PagoTarjeta(Pago):
    def cobrar(self, cantidad):
        return f"Cobrado {cantidad}€ con tarjeta"

class PagoBizum(Pago):
    def cobrar(self, cantidad):
        return f"Cobrado {cantidad}€ con Bizum"


# ---------- IMPLEMENTACIONES INCORRECTAS ----------

class PagoEfectivo(Pago):
    pass  # no implementa cobrar

class PagoTransferencia(Pago):
    def pagar(self, cantidad):  # nombre incorrecto
        return cantidad

class PagoCripto(Pago):
    def cobrar(self):  # firma incorrecta
        return "Pago en cripto"

# ---------- PRUEBAS ----------

pagos = [
    PagoTarjeta(),
    PagoBizum(),
]

for p in pagos:
    print(p.cobrar(10))

# Estas líneas FALLAN al instanciar (descomentar una a una)
# PagoEfectivo()
# PagoTransferencia()
# PagoCripto()


Cobrado 10€ con tarjeta
Cobrado 10€ con Bizum


## Trampas comunes y buenas prácticas

- Encapsulamiento no es “ocultar por ocultar”: es proteger invariantes.
- Herencia mal aplicada crea jerarquías frágiles; evita árboles profundos.
- Con polimorfismo, prefieres dependencias por **interfaz** (métodos esperados), no por tipo concreto.
- Si usas `abc`, no te quedes en lo “teórico”: úsalo para contratos claros.

### Mini-práctica (sin solución aquí)
- `Producto` con `precio` validado vía `@property` y método de descuento.
- Jerarquía `EmpleadoFijo` / `EmpleadoPorHoras` con `calcular_salario()`.